In [1]:
import papermill as pm
import numpy as np
# Optuna
#!pip install optuna
import optuna

# GRID SEACH, define a parameter space and evaluate the simulation at each point uniformly

In [ ]:
temperature        = [4e-3, 5e-3, 6e-3, 8e-3]   # mK
temperature_transv = [4e-3, 5e-3, 6e-3, 8e-3]   # mK
tau_mixing         = [15, 20, 25, 30, 35] # s
theta              = [10*np.pi/180, 15*np.pi/180, 20*np.pi/180, 25*np.pi/180] 
print(temperature, temperature_transv)


import multiprocessing as mp
import papermill as pm

def run_simulation(args):
    t1, t2, tau, angle = args

    output_name = f"Simulation_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    print(f"\n>>> Executing {output_name}")

    pm.execute_notebook(
        "Simulation.ipynb",
        f"output/notebooks/{output_notebook}_{bias}.ipynb",
        parameters={
            'Temperature'       : t1,
            'Temperature_transv': t2,
            'tau_mixing'        : tau,
            'theta'             : angle,
            'stringa'           : f"tau_{tau}s_theta_{int(angle*180/np.pi)}",
            'bias'              : "0g"
        }
    )


if __name__ == "__main__":
    # genera tutte le combinazioni (equivalente ai due for annidati)
    tasks = [(t1, t2, tau, angle) for t1 in temperature for t2 in temperature_transv for tau in tau_mixing for angle in theta]

    # numero di processi (non saturare la macchina)
    n_proc = min(len(tasks), max(1, mp.cpu_count() - 1))

    with mp.Pool(processes= 7, maxtasksperchild=1) as pool:
        pool.map(run_simulation, tasks)

# Bayesian optimization, smart search of the minimum.

In [25]:
def objective(trial):
    t1    = trial.suggest_float("Temperature", 0.5e-3, 10e-3)
    t2    = trial.suggest_float("Temperature_transv", 0.5e-3, 10e-3)
    tau   = trial.suggest_float("tau_mixing", 5, 100)
    angle = trial.suggest_float("theta", 5*np.pi/180, 180*np.pi/180)

    output_notebook = f"Simulation_tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    output_name     = f"tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_axial_{t1*1e3:.2f}mK_transv_{t2*1e3:.2f}mK"
    print(f"\n>>> Executing {output_name}")
    
    pm.execute_notebook(
        "Simulation.ipynb",
        f"output/{output_notebook}.ipynb",
        parameters={
            'Temperature'       : t1,
            'Temperature_transv': t2,
            'tau_mixing'        : tau,
            'theta'             : angle,
            'stringa'           : output_name,
            'bias'              : "0g"
        }
    )

    data = np.load("output/" + output_name)
    return float(data["metric"])

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=200)

[I 2026-01-27 20:02:38,200] A new study created in memory with name: no-name-ba42efd2-c419-4a83-8874-0ebf4cb55b12



>>> Executing tau_46.36s_theta_103_axial_7.58mK_transv_6.06mK_p_2.npz


Executing:   0%|          | 0/36 [00:00<?, ?cell/s]

[I 2026-01-27 20:08:44,462] Trial 0 finished with value: 1933.1603150673166 and parameters: {'Temperature': 0.007581671145921013, 'Temperature_transv': 0.006063486556449489, 'tau_mixing': 46.356314638016215, 'theta': 1.8089518382681722}. Best is trial 0 with value: 1933.1603150673166.



>>> Executing tau_70.26s_theta_62_axial_4.75mK_transv_2.46mK_p_2.npz


...

[I 2026-01-28 11:54:57,785] Trial 124 finished with value: 409.0373677758015 and parameters: {'Temperature': 0.0038280371069116612, 'Temperature_transv': 0.001857277433531939, 'tau_mixing': 9.520261714449024, 'theta': 0.2974252473439869}. Best is trial 122 with value: 343.14721080658154.



>>> Executing tau_10.39s_theta_18_axial_4.52mK_transv_2.49mK_p_2.npz


Executing:   0%|          | 0/36 [00:00<?, ?cell/s]

[W 2026-01-28 14:13:44,814] Trial 125 failed with parameters: {'Temperature': 0.0045175193335541585, 'Temperature_transv': 0.0024881249498035275, 'tau_mixing': 10.392909488699834, 'theta': 0.3176980945571829} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/home/adriano/.local/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/tmp/ipykernel_6315/525112353.py", line 11, in objective
    pm.execute_notebook(
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/execute.py", line 116, in execute_notebook
    nb = papermill_engines.execute_notebook_with_engine(
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/engines.py", line 48, in execute_notebook_with_engine
    return self.get_engine(engine_name).execute_notebook(nb, kernel_name, **kwargs)
  File "/home/adriano/.local/lib/python3.10/site-packages/papermill/engines.py", line 370, in e

KeyboardInterrupt: 

In [26]:
print("Best LR:", study.best_value)
print("Best params:", study.best_params)

Best LR: 343.14721080658154
Best params: {'Temperature': 0.003084017369977961, 'Temperature_transv': 0.0023900622065678597, 'tau_mixing': 10.763076448542275, 'theta': 0.3486297826814604}


# Simulation with comparison S-curve and Time Distributions
The objective function will perform a complete simulation, extracting simulated time distributions and comparing it to data time distributions. The objective function will also perform a simulation to extract the Scurve and compare it to the data. The metric will be the normalized LR of the time distributions 

In [8]:
list_biases = ['-0p75g', '0p0g', '0p5g', '0p75g', '-1p25g', '-0p37g', '0p25g', '1p25g', '-0p5g', '-0p25g']

def objective(trial):
    t1    = trial.suggest_float("Temperature", 0.5e-3, 20e-3)
    t2    = trial.suggest_float("Temperature_transv", 0.5e-3, 20e-3)
    tau   = trial.suggest_float("tau_mixing", 5, 100)
    angle = trial.suggest_float("theta", 5*np.pi/180, 180*np.pi/180)

    output_notebook = f"Simulation_tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_{t1*1e3:.2}mK_{t2*1e3:.2}mK"
    output_name     = f"tau_{tau:.2f}s_theta_{int(angle*180/np.pi)}_axial_{t1*1e3:.2f}mK_transv_{t2*1e3:.2f}mK"
    print(f"\n>>> Executing {output_name}")

    for bias in list_biases:
        pm.execute_notebook(
            "Simulation.ipynb",
            f"output/notebooks/{output_notebook}_{bias}.ipynb",
            parameters={
                'Temperature'       : t1,
                'Temperature_transv': t2,
                'tau_mixing'        : tau,
                'theta'             : angle,
                'stringa'           : output_name,
                'bias'              : bias
            }
        )

    pm.execute_notebook(
            "Metric_Worker.ipynb",
            f"output/notebooks/Metric_Worker.ipynb",
            parameters={
                'outputfile' : output_name
            }
        )

    
    data = np.load("output/" + output_name + "_0p0g.npz")  # take the LR from the 0g files output.

    LR = data["metric"]
    Chisq = data['Chisq_S']
    
    return float( LR + np.mean(Chisq) )

In [9]:
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=200)

[I 2026-02-07 16:46:08,351] A new study created in memory with name: no-name-9e7e97a8-2f04-4d10-9609-446b7f5eb43d



>>> Executing tau_73.49s_theta_153_axial_9.29mK_transv_17.57mK


Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

...

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/37 [00:00<?, ?cell/s]

Executing:   0%|          | 0/13 [00:00<?, ?cell/s]

[W 2026-02-07 17:21:32,422] Trial 0 failed with parameters: {'Temperature': 0.009292876319483459, 'Temperature_transv': 0.017570652432932446, 'tau_mixing': 73.48910583867557, 'theta': 2.680794282182226} because of the following error: PapermillExecutionError(12, 11, 'data = np.load("output/" + outputfile + "_0g.npz")\nd = dict(data)\n\nd["Chisq_S"] = np.array([ compute_segment_chisq(df_onlyramps, 2, simulation_2nd),\n                          compute_segment_chisq(df_onlyramps, 3, simulation_3rd),\n                          compute_segment_chisq(df_onlywait,  2, simulation_2nd_boiloff),\n                          compute_segment_chisq(df_onlywait,  3, simulation_3rd_boiloff)\n                        ])\n\nnp.savez("output/" + outputfile + "_0g.npz", **d)', 'FileNotFoundError', "[Errno 2] No such file or directory: 'output/tau_73.49s_theta_153_axial_9.29mK_transv_17.57mK_0g.npz'", ['\x1b---------------------------------------------------------------------------\x1b', '\x1bFileNotFoundEr

PapermillExecutionError: 
---------------------------------------------------------------------------
Exception encountered at "In [11]":
---------------------------------------------------------------------------
FileNotFoundError                         Traceback (most recent call last)
Cell In[11], line 1
----> 1 data = np.load("output/" + outputfile + "_0g.npz")
      2 d = dict(data)
      4 d["Chisq_S"] = np.array([ compute_segment_chisq(df_onlyramps, 2, simulation_2nd),
      5                           compute_segment_chisq(df_onlyramps, 3, simulation_3rd),
      6                           compute_segment_chisq(df_onlywait,  2, simulation_2nd_boiloff),
      7                           compute_segment_chisq(df_onlywait,  3, simulation_3rd_boiloff)
      8                         ])

File ~/.local/lib/python3.10/site-packages/numpy/lib/npyio.py:427, in load(file, mmap_mode, allow_pickle, fix_imports, encoding, max_header_size)
    425     own_fid = False
    426 else:
--> 427  